# BP timeline + waveform: cuff NBP (chart) vs invasive ABP (hf)

Per-patient whole-encounter view. Loads the processed `ehr_events.npy` (cuff NBP,
CHARTEVENTS, var 104/105/106) and `ehr_hf.npy` (invasive ABP + HR/SpO2/RR from the
monitor numerics, var 150-155), plus the `PLETH40`/`II120` waveforms.

**Panels:** (1) BP — invasive ABP lines vs cuff NBP markers; (2) PLETH40 envelope;
(3) II120 envelope. All share the time axis.

**hf temporal coverage (400-patient sample):** HR/SpO2/RR ~100% of patients, a value
at ~every 2-seg pair, whole stay. Invasive ABP only ~32% of patients (arterial line);
when present it spans ~81% of the stay, ~66% of pairs filled.

In [ ]:
import os, json, warnings
import numpy as np
import matplotlib.pyplot as plt

ROOT     = "/opt/localdata100tb/physio_data/mimic3"
REGISTRY = "/labs/hulab/mxwang/Physio_Data/indices/var_registry.json"
ENTITY   = "51805_185552"          # <-- change me (Cell 4 lists candidates)

NBP    = {104: ("NBPs", "chart"), 105: ("NBPd", "chart"), 106: ("NBPm", "chart")}   # ehr_events.npy
ABP    = {153: ("ABPs_hf", "hf"), 154: ("ABPd_hf", "hf"), 155: ("ABPm_hf", "hf")}   # ehr_hf.npy
HF_ALL = {150: "HR", 151: "SpO2", 152: "RR", 153: "ABPs_hf", 154: "ABPd_hf", 155: "ABPm_hf"}
COMPONENT = {104:"sys",105:"dias",106:"mean", 153:"sys",154:"dias",155:"mean"}
COLOR = {"sys": "#d62728", "dias": "#1f77b4", "mean": "#2ca02c"}
CH_RATE = {"PLETH40": 40, "II120": 120}     # Hz -> samples/30s = rate*30

In [ ]:
def load_entity(entity):
    d = os.path.join(ROOT, entity)
    ev = np.load(os.path.join(d, "ehr_events.npy"))
    hp = os.path.join(d, "ehr_hf.npy")
    hf = np.load(hp) if os.path.exists(hp) else np.empty(0, dtype=ev.dtype)
    meta = json.load(open(os.path.join(d, "meta.json")))
    time_ms = np.load(os.path.join(d, "time_ms.npy"))
    return ev, hf, meta, time_ms

def series(events, var_id, t0_ms):
    sel = events[events["var_id"] == var_id]
    sel = sel[np.argsort(sel["time_ms"])]
    return (sel["time_ms"].astype(np.float64) - t0_ms) / 3.6e6, sel["value"].astype(np.float64)

def seg_envelope(entity, channel, gap_ms=45000):
    """Per-segment min/max of a waveform vs hours-from-start, NaN-broken at recording gaps.
    A whole-timeline 'where is the signal + how pulsatile' view (raw values, not normalized)."""
    d = os.path.join(ROOT, entity)
    p = os.path.join(d, channel + ".npy")
    if not os.path.exists(p):
        return None
    w = np.asarray(np.load(p, mmap_mode="r"), dtype=np.float32)     # [N_seg, S]
    tm = np.load(os.path.join(d, "time_ms.npy"))
    with warnings.catch_warnings():                                 # all-NaN segs -> NaN (channel absent)
        warnings.simplefilter("ignore", RuntimeWarning)
        lo = np.nanmin(w, axis=1); hi = np.nanmax(w, axis=1)
    t_h = (tm.astype(np.float64) - tm[0]) / 3.6e6
    cut = np.where(np.diff(tm) > gap_ms)[0] + 1                     # break the fill across gaps
    t_h = np.insert(t_h, cut, np.nan); lo = np.insert(lo, cut, np.nan); hi = np.insert(hi, cut, np.nan)
    return t_h, lo, hi

In [ ]:
# --- OPTIONAL: candidates that have BOTH cuff NBP and invasive ABP ---
def find_patients_with_both(n=10, min_nbp=5, min_abp=200, scan=800):
    sf = json.load(open(os.path.join(ROOT, "tasks/vital_est_full/splits.json")))
    ents = sf["train"] + sf["val"] + sf["test"]
    out = []
    for e in ents[:scan]:
        d = os.path.join(ROOT, e)
        if not os.path.exists(os.path.join(d, "ehr_hf.npy")):
            continue
        ev = np.load(os.path.join(d, "ehr_events.npy")); hf = np.load(os.path.join(d, "ehr_hf.npy"))
        nn = int(np.isin(ev["var_id"], list(NBP)).sum()); na = int(np.isin(hf["var_id"], list(ABP)).sum())
        if nn >= min_nbp and na >= min_abp:
            out.append((e, nn, na))
        if len(out) >= n:
            break
    return out

for e, nn, na in find_patients_with_both():
    print(f"{e:18s}  NBP(chart)={nn:5d}  ABP_hf={na:6d}")

In [ ]:
# --- MAIN: BP + waveform envelopes, whole encounter, shared time axis ---
ev, hf, meta, time_ms = load_entity(ENTITY)
t0 = int(time_ms[0]); dur_h = (int(time_ms[-1]) - t0) / 3.6e6

fig, (axb, axp, axi) = plt.subplots(
    3, 1, figsize=(16, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1.2, 1.2]})

# (1) BP
for vid, (lab, _) in ABP.items():
    t, v = series(hf, vid, t0)
    if len(t): axb.plot(t, v, "-", lw=0.8, alpha=0.9, color=COLOR[COMPONENT[vid]],
                        label=f"{lab} (invasive, n={len(t)})")
for vid, (lab, _) in NBP.items():
    t, v = series(ev, vid, t0)
    if len(t): axb.plot(t, v, "o", ms=6, mfc="none", mew=1.6, color=COLOR[COMPONENT[vid]],
                        label=f"{lab} (cuff, n={len(t)})")
axb.set_ylabel("BP (mmHg)"); axb.set_ylim(0, 220); axb.grid(alpha=.3)
axb.legend(ncol=2, fontsize=8, loc="upper right")
axb.set_title(f"{ENTITY}  |  invasive ABP (1/min) vs cuff NBP (~hourly) + waveform  |  "
              f"{dur_h:.1f} h, {meta.get('n_segments')} segs")

# (2)(3) waveform envelopes (per-segment min-max band)
for ax, ch, color in [(axp, "PLETH40", "#8c564b"), (axi, "II120", "#7f7f7f")]:
    r = seg_envelope(ENTITY, ch)
    if r is not None:
        th, lo, hi = r
        ax.fill_between(th, lo, hi, color=color, alpha=0.6, lw=0)
    ax.set_ylabel(f"{ch}\n(min-max/seg)"); ax.grid(alpha=.3)
axi.set_xlabel("hours from recording start")
plt.tight_layout(); plt.show()

In [ ]:
# --- hf temporal coverage for THIS patient (fraction of 2-seg pairs with a value) ---
ev, hf, meta, time_ms = load_entity(ENTITY)
t0 = int(time_ms[0]); dur_h = (int(time_ms[-1]) - t0) / 3.6e6
n_pairs = max(1, len(time_ms) // 2)
print(f"{ENTITY}: {meta.get('n_segments')} segs -> {n_pairs} 2-seg pairs, {dur_h:.1f} h\n")
print(f"{'var':9s}{'n_events':>9s}{'cov(of pairs)':>15s}{'time-span':>11s}")
for vid, lab in HF_ALL.items():
    s = hf[hf["var_id"] == vid]
    span = ((int(s["time_ms"].max()) - int(s["time_ms"].min())) / max(1, int(time_ms[-1]) - t0)) if len(s) else 0.0
    print(f"{lab:9s}{len(s):9d}{len(s)/n_pairs:15.2f}{span:11.2f}")

In [ ]:
# --- RAW waveform ZOOM: inspect actual PPG/ECG shape in a short window ---
ZOOM_START_H = 10.0     # window start (hours from recording start)
ZOOM_LEN_MIN = 2.0      # window length (minutes)

d = os.path.join(ROOT, ENTITY); tm = np.load(os.path.join(d, "time_ms.npy")); t0 = int(tm[0])
s0 = int(np.searchsorted(tm, t0 + ZOOM_START_H * 3.6e6))
s1 = int(np.searchsorted(tm, t0 + ZOOM_START_H * 3.6e6 + ZOOM_LEN_MIN * 60e3))
s1 = max(s1, s0 + 1)

fig, axes = plt.subplots(2, 1, figsize=(16, 5), sharex=True)
for ax, ch in zip(axes, ["PLETH40", "II120"]):
    p = os.path.join(d, ch + ".npy")
    if not os.path.exists(p):
        continue
    sig = np.asarray(np.load(p, mmap_mode="r")[s0:s1], dtype=np.float32).reshape(-1)  # 30s non-overlap -> continuous
    fs = CH_RATE[ch]
    t_min = (np.arange(len(sig)) / fs + (int(tm[s0]) - t0) / 1000.0) / 60.0
    ax.plot(t_min, sig, lw=0.5); ax.set_ylabel(ch); ax.grid(alpha=.3)
axes[-1].set_xlabel("minutes from recording start")
axes[0].set_title(f"{ENTITY} raw waveform  [{ZOOM_START_H} h + {ZOOM_LEN_MIN} min]")
plt.tight_layout(); plt.show()

**Reading it**

- **BP panel:** solid = invasive arterial line (dense, 1/min); open markers = cuff
  (~hourly). sys (red) > mean (green) > dias (blue). ABP lines exist only while the
  arterial line was in.
- **Waveform panels:** per-segment min-max band = where the PPG/ECG exists and how
  pulsatile it is; breaks = recording gaps; II120 blank stretches = ECG lead absent.
- **Coverage cell:** `cov` = fraction of 2-seg pairs with a value (HR/SpO2/RR ~1.0;
  ABP partial, arterial-line window only).
- **Zoom cell:** raw waveform shape; move `ZOOM_START_H` into an arterial-line stretch
  to compare the PPG pulse against the invasive ABP at that time.